<a href="https://colab.research.google.com/github/serenalxs/Practice/blob/main/MultiLayer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import torch
from torch import nn

In [2]:
url = 'https://raw.githubusercontent.com/serenalxs/Practice/refs/heads/main/Basic/HR.csv'
data = pd.read_csv(url)
data.info()

data.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14999 entries, 0 to 14998
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   satisfaction_level     14999 non-null  float64
 1   last_evaluation        14999 non-null  float64
 2   number_project         14999 non-null  int64  
 3   average_montly_hours   14999 non-null  int64  
 4   time_spend_company     14999 non-null  int64  
 5   Work_accident          14999 non-null  int64  
 6   left                   14999 non-null  int64  
 7   promotion_last_5years  14999 non-null  int64  
 8   part                   14999 non-null  object 
 9   salary                 14999 non-null  object 
dtypes: float64(2), int64(6), object(2)
memory usage: 1.1+ MB


,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,part,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,0.72,0.87,5,223,5,0,1,0,sales,low
4,0.37,0.52,2,159,3,0,1,0,sales,low


In [3]:
print (data.part.unique() )
print (data.salary.unique())
print (data.groupby(['salary','part']).size())

['sales' 'accounting' 'hr' 'technical' 'support' 'management' 'IT'
 'product_mng' 'marketing' 'RandD']
['low' 'medium' 'high']
salary  part       
high    IT               83
        RandD            51
        accounting       74
        hr               45
        management      225
        marketing        80
        product_mng      68
        sales           269
        support         141
        technical       201
low     IT              609
        RandD           364
        accounting      358
        hr              335
        management      180
        marketing       402
        product_mng     451
        sales          2099
        support        1146
        technical      1372
medium  IT              535
        RandD           372
        accounting      335
        hr              359
        management      225
        marketing       376
        product_mng     383
        sales          1772
        support         942
        technical      1147
dtype: int64


In [4]:
# dummy variable;

data = data.join(pd.get_dummies(data.salary).astype(int))
data = data.join(pd.get_dummies(data.part).astype(int))
del data['salary']
del data['part']
print(data.head(5))

   satisfaction_level  last_evaluation  number_project  average_montly_hours  \
0                0.38             0.53               2                   157   
1                0.80             0.86               5                   262   
2                0.11             0.88               7                   272   
3                0.72             0.87               5                   223   
4                0.37             0.52               2                   159   

   time_spend_company  Work_accident  left  promotion_last_5years  high  low  \
0                   3              0     1                      0     0    1   
1                   6              0     1                      0     0    0   
2                   4              0     1                      0     0    0   
3                   5              0     1                      0     0    1   
4                   3              0     1                      0     0    1   

   ...  IT  RandD  accounting  hr  man

In [5]:
print (data.shape)
print("--------attribute: left----------")
print (data.left.value_counts())
Y = torch.from_numpy(data.left.values.reshape(-1,1)).type(torch.float32)
print("Y shape:", Y.shape)
X_data = data[[ c for c in data.columns if c != 'left']].values
X = torch.from_numpy(X_data).type(torch.float32)
print ("X shape:" , X.shape)

(14999, 21)
--------attribute: left----------
left
0    11428
1     3571
Name: count, dtype: int64
Y shape: torch.Size([14999, 1])
X shape: torch.Size([14999, 20])


# **Model**

nn.model : inheritate model

init : initialize

forward : define model

In [6]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_1 = nn.Linear(20,64)
        self.linear_2 = nn.Linear(64,64)
        self.linear_3 = nn.Linear(64,1)
        self.relu     = nn.ReLU()
        self.sigmoid  = nn.Sigmoid()

    def forward(self,input):
        x = self.linear_1(input)
        x = self.relu(x)
        x = self.linear_2(x)
        x = self.relu(x)
        x = self.linear_3(x)
        x = self.sigmoid(x)
        return x

def get_model():
    model = Model()
    opt   = torch.optim.Adam(model.parameters(),lr = 0.0001)
    return model,opt

model = Model()
model , optim = get_model()


print(model)
print(optim)

Model(
  (linear_1): Linear(in_features=20, out_features=64, bias=True)
  (linear_2): Linear(in_features=64, out_features=64, bias=True)
  (linear_3): Linear(in_features=64, out_features=1, bias=True)
  (relu): ReLU()
  (sigmoid): Sigmoid()
)
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0
)


In [7]:
model = Model()
model , optim = get_model()

lr = 0.0001
loss_fn = nn.BCELoss()
batches = 64
no_of_batches = len(X) // batches
epochs = 100

for epoch in range(5):
    for i in range(no_of_batches):
        start = i*batches
        end   = i*batches + batches
        x     = X[start:end]
        y     = Y[start:end]
        y_pred= model(x)
        loss  = loss_fn(y_pred,y)
        optim.zero_grad()
        loss.backward()
        optim.step()
    with torch.no_grad():
        loss_fn(model(X),Y)
        print("epoch:\t",epoch,"\t\t\t","loss:\t",loss_fn(model(X),Y).data.item())


epoch:	 0 			 loss:	 0.6790480613708496
epoch:	 1 			 loss:	 0.6691790223121643
epoch:	 2 			 loss:	 0.67320716381073
epoch:	 3 			 loss:	 0.693788468837738
epoch:	 4 			 loss:	 0.6329291462898254


# **better way to create batches : torch.dataset **

dataset : auto slice data to batch
dataloader : reshuffle data batches

In [8]:
from torch.utils.data import TensorDataset, DataLoader

HRdataset = TensorDataset(X,Y)

print(HRdataset[0])

print(HRdataset[1:5])

(tensor([  0.3800,   0.5300,   2.0000, 157.0000,   3.0000,   0.0000,   0.0000,
          0.0000,   1.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   1.0000,   0.0000,   0.0000]), tensor([1.]))
(tensor([[8.0000e-01, 8.6000e-01, 5.0000e+00, 2.6200e+02, 6.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00,
         0.0000e+00, 0.0000e+00],
        [1.1000e-01, 8.8000e-01, 7.0000e+00, 2.7200e+02, 4.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00,
         0.0000e+00, 0.0000e+00],
        [7.2000e-01, 8.7000e-01, 5.0000e+00, 2.2300e+02, 5.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+0

In [9]:
model , optim = get_model()  #every time build a new model
HRds = TensorDataset(X,Y)
HRdl = DataLoader(HRds, batch_size = batches, shuffle =True)
for epoch in range(epochs) :
  for x ,  y in HRdl :
    y_pred = model(x)

    loss = loss_fn (y_pred , y)

    optim.zero_grad()
    loss.backward()
    optim.step()

  with torch.no_grad() :
    print ("epoch:", epoch)
    print ("loss:", loss_fn(model(X), Y).data.item())

epoch: 0
loss: 0.5610913038253784
epoch: 1
loss: 0.5571794509887695
epoch: 2
loss: 0.5559373497962952
epoch: 3
loss: 0.5499217510223389
epoch: 4
loss: 0.5533422827720642
epoch: 5
loss: 0.5396496057510376
epoch: 6
loss: 0.533363401889801
epoch: 7
loss: 0.5305560827255249
epoch: 8
loss: 0.5218920111656189
epoch: 9
loss: 0.5139734745025635
epoch: 10
loss: 0.5171253681182861
epoch: 11
loss: 0.5023320317268372
epoch: 12
loss: 0.49168574810028076
epoch: 13
loss: 0.49645838141441345
epoch: 14
loss: 0.4773064851760864
epoch: 15
loss: 0.46888792514801025
epoch: 16
loss: 0.4759856164455414
epoch: 17
loss: 0.4543944001197815
epoch: 18
loss: 0.4497557580471039
epoch: 19
loss: 0.44343316555023193
epoch: 20
loss: 0.4337300658226013
epoch: 21
loss: 0.42736899852752686
epoch: 22
loss: 0.42487633228302
epoch: 23
loss: 0.41595888137817383
epoch: 24
loss: 0.4132842719554901
epoch: 25
loss: 0.405432790517807
epoch: 26
loss: 0.4016783535480499
epoch: 27
loss: 0.39541423320770264
epoch: 28
loss: 0.391481459

# **Evaluate the model**

Overfitting vs underfitting

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
train_X, test_X, train_Y,  test_Y = train_test_split (X, Y, test_size = 0.25)

print(train_X.shape)
print(train_Y.shape)
print(test_X.shape)
print(test_Y.shape)

train_ds = TensorDataset (train_X, train_Y)
test_ds = TensorDataset (test_X, test_Y)

train_dl = DataLoader (train_ds, batch_size = batches, shuffle =True)
test_dl = DataLoader (test_ds, batch_size = batches, shuffle =False ) #test data no need to shuffle




torch.Size([11249, 20])
torch.Size([11249, 1])
torch.Size([3750, 20])
torch.Size([3750, 1])


In [12]:
train_dl_iter = iter(train_dl)
first_batch = next(train_dl_iter)
print("First batch:", first_batch)
second_batch = next(train_dl_iter)
print("Second batch:", second_batch)

First batch: [tensor([[0.9800, 0.7500, 5.0000,  ..., 1.0000, 0.0000, 0.0000],
        [0.3800, 1.0000, 5.0000,  ..., 0.0000, 1.0000, 0.0000],
        [0.9600, 0.7500, 5.0000,  ..., 1.0000, 0.0000, 0.0000],
        ...,
        [0.1600, 0.7200, 4.0000,  ..., 1.0000, 0.0000, 0.0000],
        [0.6000, 0.7700, 4.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.5500, 0.9700, 5.0000,  ..., 1.0000, 0.0000, 0.0000]]), tensor([[0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],

Accuracy

In [16]:
from torch import nn
import torch.nn.functional as F

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_1 = nn.Linear(20,64)
        self.linear_2 = nn.Linear(64,64)
        self.linear_3 = nn.Linear(64,1)

    def forward(self,input):
        x = self.linear_1(input)
        x = F.relu(x)
        x = self.linear_2(x)
        x = F.relu(x)
        x = self.linear_3(x)
        x = F.sigmoid(x)
        return x

In [19]:
def accuracy(y_predict,y_true):
    y_predict = (y_predict>0.5).type(torch.int32)
    acc       = (y_predict == y_true).float().mean()
    return acc


In [24]:
model , optim = get_model()

lr = 0.0001
loss_fn = nn.BCELoss()
epochs = 100

for epoch in range(epochs):
    for x,y in train_dl:
        y_pred = model(x)
        loss = loss_fn(y_pred,y)
        optim.zero_grad()
        loss.backward()
        optim.step()
    with torch.no_grad():
        epoch_accuracy = accuracy(model(train_X),train_Y)
        epoch_loss = loss_fn(model(train_X),train_Y).data

        epoch_test_accuracy = accuracy(model(test_X),test_Y)
        epoch_test_loss = loss_fn(model(test_X),test_Y).data

        print("epoch:","\t",epoch,"\t","loss: ","\t",round(epoch_loss.item(),3),"\t",
                             "accuracy: ","\t",round(epoch_accuracy.item(),3),"\t",
                             "test_loss: ","\t",round(epoch_test_loss.item(),3),"\t",
                             "test_accuracy: ","\t",round(epoch_test_accuracy.item(),3),"\t"
             )

epoch: 	 0 	 loss:  	 0.563 	 accuracy:  	 0.763 	 test_loss:  	 0.569 	 test_accuracy:  	 0.759 	
epoch: 	 1 	 loss:  	 0.561 	 accuracy:  	 0.763 	 test_loss:  	 0.567 	 test_accuracy:  	 0.759 	
epoch: 	 2 	 loss:  	 0.556 	 accuracy:  	 0.763 	 test_loss:  	 0.562 	 test_accuracy:  	 0.759 	
epoch: 	 3 	 loss:  	 0.551 	 accuracy:  	 0.763 	 test_loss:  	 0.557 	 test_accuracy:  	 0.759 	
epoch: 	 4 	 loss:  	 0.547 	 accuracy:  	 0.763 	 test_loss:  	 0.553 	 test_accuracy:  	 0.759 	
epoch: 	 5 	 loss:  	 0.542 	 accuracy:  	 0.763 	 test_loss:  	 0.547 	 test_accuracy:  	 0.759 	
epoch: 	 6 	 loss:  	 0.537 	 accuracy:  	 0.763 	 test_loss:  	 0.542 	 test_accuracy:  	 0.759 	
epoch: 	 7 	 loss:  	 0.53 	 accuracy:  	 0.763 	 test_loss:  	 0.536 	 test_accuracy:  	 0.759 	
epoch: 	 8 	 loss:  	 0.529 	 accuracy:  	 0.763 	 test_loss:  	 0.536 	 test_accuracy:  	 0.759 	
epoch: 	 9 	 loss:  	 0.516 	 accuracy:  	 0.763 	 test_loss:  	 0.522 	 test_accuracy:  	 0.759 	
epoch: 	 10